# Pipeline de Dados - MVP E-Commerce

Descrição:
- Este notebook contém a implementação da ingestão (Camada Bronze); 
- Limpeza e transformação (Camada Silver);
- Modelagem analítica (Camada Gold) conforme os requisitos do projeto.

## Camada Bronze
Ingestão dos dados obtidos no Kaggle e inseridos no databricks.

Link do dataset utilizado: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data

### Orders dataset

In [0]:
# Importando Funções do SQL no PySpark
from pyspark.sql import functions as F

### Orders dataset

In [0]:
# 1. Caminho do arquivo bruto no Unity Catalog Volumes do Databricks
file_path = "dbfs:/Volumes/pos_puc_rio/mvp_ecommerce/dados_bruto/archive/olist_orders_dataset.csv"

# 2. Leitura dos dados brutos
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 3. Adicionar metadados de controle da Camada Bronze
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
                  .withColumn("source_file", F.lit("olist_orders_dataset.csv"))

# 4. Persistir na Camada Bronze (Tabela Delta)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")

print("Camada Bronze gerada com sucesso!")
display(spark.sql("SELECT * FROM bronze_orders LIMIT 5"))

### Orders items dataset

In [0]:
# 1. Caminho do arquivo bruto no Unity Catalog Volumes do Databricks
file_path = "dbfs:/Volumes/pos_puc_rio/mvp_ecommerce/dados_bruto/archive/olist_order_items_dataset.csv"

# 2. Leitura dos dados brutos
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 3. Adicionar metadados de controle da Camada Bronze
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
                  .withColumn("source_file", F.lit("olist_order_items_dataset.csv"))

# 4. Persistir na Camada Bronze (Tabela Delta)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_order_items")

print("Camada Bronze gerada com sucesso!")
display(spark.sql("SELECT * FROM bronze_order_items LIMIT 5"))

### Products dataset

In [0]:
# 1. Caminho do arquivo bruto no Unity Catalog Volumes do Databricks
file_path = "dbfs:/Volumes/pos_puc_rio/mvp_ecommerce/dados_bruto/archive/olist_products_dataset.csv"

# 2. Leitura dos dados brutos
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 3. Adicionar metadados de controle da Camada Bronze
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
                  .withColumn("source_file", F.lit("olist_products_dataset.csv"))

# 4. Persistir na Camada Bronze (Tabela Delta)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_products")

print("Camada Bronze gerada com sucesso!")
display(spark.sql("SELECT * FROM bronze_products LIMIT 5"))

### Costumers dataset

In [0]:
# 1. Caminho do arquivo bruto no Unity Catalog Volumes do Databricks
file_path = "dbfs:/Volumes/pos_puc_rio/mvp_ecommerce/dados_bruto/archive/olist_customers_dataset.csv"

# 2. Leitura dos dados brutos
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 3. Adicionar metadados de controle da Camada Bronze
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
                  .withColumn("source_file", F.lit("olist_customers_dataset.csv"))

# 4. Persistir na Camada Bronze (Tabela Delta)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")

print("Camada Bronze gerada com sucesso!")
display(spark.sql("SELECT * FROM bronze_products LIMIT 5"))

### Order Payments dataset

In [0]:
# 1. Caminho do arquivo bruto no Unity Catalog Volumes do Databricks
file_path = "dbfs:/Volumes/pos_puc_rio/mvp_ecommerce/dados_bruto/archive/olist_order_payments_dataset.csv"

# 2. Leitura dos dados brutos
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 3. Adicionar metadados de controle da Camada Bronze
df_bronze = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
                  .withColumn("source_file", F.lit("olist_order_payments_dataset.csv"))

# 4. Persistir na Camada Bronze (Tabela Delta)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_order_payments")

print("Camada Bronze gerada com sucesso!")
display(spark.sql("SELECT * FROM bronze_order_payments LIMIT 5"))

## Camada Silver
### Análise de Qualidade e Tratamento
Nesta etapa, verificamos a integridade, valores nulos e duplicatas da tabela Bronze para gerar a tabela limpa Silver.

#### Transformação da camada bronze de ORDERS em silver

In [0]:
# Exemplo de tratamento no data frame para a camada Silver (remover nulos [dropna] e duplicatas [dropDuplicates])
df_silver = spark.table("bronze_orders") \
    .dropna(subset=["order_id", "customer_id"]) \
    .dropDuplicates(["order_id"])

# Adicionar tratamento de datas (exemplo)
df_silver = df_silver.withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))

# Persistir na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

print("Camada Silver gerada com sucesso!")
display(spark.sql("SELECT * FROM silver_orders LIMIT 5"))

#### Quarentena da camada bronze de ORDERS que não avançou para a camada _silver_

In [0]:
#Aplicando o filtro de valores nulos e aplicando o motivo de não importação
df_quarantine = spark.table("bronze_orders")
df_quarantine = df_quarantine.filter(F.col("order_id").isNull() | F.col("customer_id").isNull())\
.withColumn("error_reason", F.lit("Missing mandatory primary keys (order_id or customer_id)"))\
.withColumn("quarantine_timestamp", F.current_timestamp())

df_quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_orders")

display(spark.sql("SELECT * FROM quarantine_orders LIMIT 5"))

#### Transformação da camada bronze de ORDER ITEMS em silver

In [0]:
# Exemplo de tratamento no data frame para a camada Silver (remover nulos [dropna])

df_silver = spark.table("bronze_order_items") \
    .dropna(subset=["order_id", "product_id", "price"])

# Adicionar tratamento de datas
df_silver = df_silver.withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))

# Adicionar tratamento de valores reais
df_silver = df_silver.withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

# Persistir na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_order_items")

print("Camada Silver gerada com sucesso!")
display(spark.sql("SELECT * FROM silver_order_items LIMIT 5"))

#### Quarentena da camada bronze de ORDERS ITEMS que não avançou para a camada _silver_

In [0]:
#Aplicando o filtro de valores nulos e aplicando o motivo de não importação
df_quarantine = spark.table("bronze_order_items")
df_quarantine = df_quarantine.filter(F.col("order_id").isNull() | F.col("product_id").isNull() | F.col("price").isNull())\
.withColumn("error_reason", F.lit("Missing mandatory keys (order_id, product_id or price)"))\
.withColumn("quarantine_timestamp", F.current_timestamp())

df_quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_orders_items")

display(spark.sql("SELECT * FROM quarantine_orders_items LIMIT 5"))

#### Transformação da camada bronze de PRODUCTS em silver

In [0]:
# Exemplo de tratamento no data frame para a camada Silver (remover nulos [dropna])

df_silver = spark.table("bronze_products") \
    .dropna(subset=["product_id"])

# Evita perda produtos em joins futuros preenchendo as categorias que estiverem vazias
df_silver = df_silver.fillna({"product_category_name": "desconhecido"})

# Persistir na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_products")

print("Camada Silver gerada com sucesso!")
display(spark.sql("SELECT * FROM silver_products LIMIT 5"))

#### Quarentena da camada bronze de PRODUCTS que não avançou para a camada _silver_

In [0]:
#Aplicando o filtro de valores nulos e aplicando o motivo de não importação
df_quarantine = spark.table("bronze_products")
df_quarantine = df_quarantine.filter(F.col("product_id").isNull())\
.withColumn("error_reason", F.lit("Missing product_id"))\
.withColumn("quarantine_timestamp", F.current_timestamp())

df_quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_products")

display(spark.sql("SELECT * FROM quarantine_products LIMIT 5"))

#### Transformação da camada bronze de CUSTOMERS em silver

In [0]:
# Exemplo de tratamento no data frame para a camada Silver (remover nulos [dropna])

df_silver = spark.table("bronze_customers") \
    .dropna(subset=["customer_id", "customer_unique_id"])

# Padronização de strings (remover espaços, colocar estados em maiúsculas e cidades com iniciais maiúsculas)
df_silver = df_silver.withColumn("customer_state", F.upper(F.trim(F.col("customer_state")))) \
    .withColumn("customer_city", F.initcap(F.trim(F.col("customer_city"))))

# Persistir na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

print("Camada Silver gerada com sucesso!")
display(spark.sql("SELECT * FROM silver_customers LIMIT 5"))

#### Quarentena da camada bronze de CUSTOMERS que não avançou para a camada _silver_

In [0]:
#Aplicando o filtro de valores nulos e aplicando o motivo de não importação
df_quarantine = spark.table("bronze_customers")
df_quarantine = df_quarantine.filter(F.col("customer_id").isNull() | F.col("customer_unique_id").isNull())\
.withColumn("error_reason", F.lit("Missing mandatory customer IDs"))\
.withColumn("quarantine_timestamp", F.current_timestamp())

df_quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_customers")

display(spark.sql("SELECT * FROM quarantine_customers LIMIT 5"))

#### Transformação da camada bronze de ORDER PAYMENTS em silver

In [0]:
# Exemplo de tratamento no data frame para a camada Silver (remover nulos [dropna])

df_silver = spark.table("bronze_order_payments") \
    .dropna(subset=["order_id", "payment_type"])

# Padronização do tipo de pagamento
df_silver = df_silver.withColumn("payment_type", F.initcap(F.regexp_replace(F.trim(F.col("payment_type")), "_", " "))) \
    .withColumn("payment_type", F.regexp_replace(F.col("payment_type"), "Credit Card", "Cartao de Credito")) \
    .withColumn("payment_type", F.regexp_replace(F.col("payment_type"), "Debit Card", "Cartao de Debito"))


# Tipagem para valores e parcelas
df_silver = df_silver.withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .withColumn("payment_value", F.col("payment_value").cast("double"))

# Persistir na Camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_order_payments")

print("Camada Silver gerada com sucesso!")
display(spark.sql("SELECT * FROM silver_order_payments LIMIT 5"))

#### Quarentena da camada bronze de ORDERS PAYMENTS que não avançou para a camada _silver_

In [0]:
#Aplicando o filtro de valores nulos e aplicando o motivo de não importação
df_quarantine = spark.table("bronze_order_payments")
df_quarantine = df_quarantine.filter(F.col("order_id").isNull() | F.col("payment_type").isNull()
)\
.withColumn("error_reason", F.lit("Missing order_id or payment_type"))\
.withColumn("quarantine_timestamp", F.current_timestamp())

df_quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_order_payments")

display(spark.sql("SELECT * FROM quarantine_order_payments LIMIT 5"))

## Camada Gold
### Modelagem Analítica e Respostas de Negócio
Nesta etapa, criamos as tabelas agregadas e o Esquema Estrela (Camada Gold) para responder diretamente às perguntas de negócio delineadas no início do projeto.

#### PERGUNTA 1 - Categoria de produto com maior receita total

In [0]:
df_gold_receita_categoria = spark.sql("""
    SELECT 
        p.product_category_name AS categoria,
        COUNT(DISTINCT o.order_id) AS total_pedidos,
        ROUND(SUM(oi.price), 2) AS receita_total
    FROM silver_orders o
    JOIN silver_order_items oi ON o.order_id = oi.order_id
    JOIN silver_products p ON oi.product_id = p.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
    ORDER BY receita_total DESC
""")

df_gold_receita_categoria.write.format("delta").mode("overwrite").saveAsTable("gold_receita_por_categoria")
print("--- Pergunta 1: Receita por Categoria ---")
display(df_gold_receita_categoria.limit(5))

#### PERGUNTA 2 - Correlação entre frete e tempo de entrega por região

In [0]:
df_gold_frete_regiao = spark.sql("""
    SELECT 
        c.customer_state AS estado,
        ROUND(AVG(oi.freight_value), 2) AS media_frete,
        ROUND(AVG(DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp)), 0) AS media_dias_entrega,
        COUNT(DISTINCT o.order_id) AS total_pedidos
    FROM silver_orders o
    JOIN silver_customers c ON o.customer_id = c.customer_id
    JOIN silver_order_items oi ON o.order_id = oi.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL 
      AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY media_dias_entrega DESC
""")

df_gold_frete_regiao.write.format("delta").mode("overwrite").saveAsTable("gold_frete_tempo_regiao")
print("--- Pergunta 2: Frete e Tempo de Entrega por Região ---")
display(df_gold_frete_regiao.limit(10))

#### PERGUNTA 3 - Ticket médio agrupado por método de pagamento

In [0]:
df_gold_ticket_pagamento = spark.sql("""
    SELECT 
        p.payment_type AS metodo_pagamento,
        COUNT(DISTINCT p.order_id) AS total_transacoes,
        ROUND(SUM(p.payment_value), 2) AS valor_total_transacionado,
        ROUND(AVG(p.payment_value), 2) AS ticket_medio
    FROM silver_order_payments p
    GROUP BY p.payment_type
    ORDER BY ticket_medio DESC
""")

df_gold_ticket_pagamento.write.format("delta").mode("overwrite").saveAsTable("gold_ticket_medio_pagamento")
print("--- Pergunta 3: Ticket Médio por Método de Pagamento ---")
display(df_gold_ticket_pagamento)